In [2]:
# Train a tuned random forest classifier on merged_data.csv and save the fitted model to disk
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Load cleaned merged data containing time features and lag features
file_path = 'data/merged_data.csv'
df = pd.read_csv(file_path, low_memory=False)

# Ensure datetime and sort by country + time before modeling
if 'calendar_start_date' in df.columns:
    df['calendar_start_date'] = pd.to_datetime(df['calendar_start_date'], errors='coerce')
    df = df.sort_values(['ISO_A0', 'calendar_start_date'])

# Build a binary outbreak label for classification
# 0 = no reported dengue cases, 1 = dengue cases reported
if 'dengue_total' not in df.columns:
    raise ValueError('Expected target column dengue_total in merged_data.csv')
df['outbreak_flag'] = (df['dengue_total'] > 0).astype(int)

# Select numeric and lag features based on the main notebook preprocessing pattern
feature_cols = [
    'ISO_A0',
    'year',
    'month',
    'dayofyear',
    'weekofyear',
    'lag_1',
    'lag_2',
    'lag_3',
    'temperature_c',
    'precipitation_mm'
]

# Keep only rows with complete features
model_df = df[feature_cols + ['outbreak_flag']].copy()
model_df = model_df.dropna()

# Encode country codes and scale numeric features
encoder = LabelEncoder()
model_df['ISO_A0_encoded'] = encoder.fit_transform(model_df['ISO_A0'])

numeric_cols = ['year', 'month', 'dayofyear', 'weekofyear', 'lag_1', 'lag_2', 'lag_3', 'temperature_c', 'precipitation_mm']
scaler = StandardScaler()
model_df[numeric_cols] = scaler.fit_transform(model_df[numeric_cols])

X = model_df[['ISO_A0_encoded'] + numeric_cols]
y = model_df['outbreak_flag']

# Train/test split with stratification on the binary target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Grid search for a tuned random forest classifier
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

rf = RandomForestClassifier(random_state=42, class_weight='balanced')
search = GridSearchCV(
    rf,
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
search.fit(X_train, y_train)

best_model = search.best_estimator_
print('Best hyperparameters:')
print(search.best_params_)
print(f'Best CV F1 score: {search.best_score_:.4f}')

# Evaluate on the holdout test set
y_pred = best_model.predict(X_test)
print('\nTest set results:')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 score: {f1_score(y_test, y_pred):.4f}')
print('Classification report:')
print(classification_report(y_test, y_pred, digits=4))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

# Persist the tuned model and preprocessing objects
joblib.dump({
    'model': best_model,
    'label_encoder': encoder,
    'scaler': scaler,
    'feature_columns': X.columns.tolist()
}, 'random_forest_outbreak_model.joblib')
print('Saved tuned random forest model to random_forest_outbreak_model.joblib')

Fitting 3 folds for each of 48 candidates, totalling 144 fits
Best hyperparameters:
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best CV F1 score: 0.9586

Test set results:
Accuracy: 0.9231
F1 score: 0.9591
Classification report:
              precision    recall  f1-score   support

           0     0.6255    0.2409    0.3478       714
           1     0.9332    0.9866    0.9591      7671

    accuracy                         0.9231      8385
   macro avg     0.7793    0.6137    0.6535      8385
weighted avg     0.9070    0.9231    0.9071      8385

Confusion matrix:
[[ 172  542]
 [ 103 7568]]
Saved tuned random forest model to random_forest_outbreak_model.joblib
